# 第3部 ニューラルネット：距離と保持

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Asg1223/rcj-soccer-ml/blob/main/notebooks/03_neural_net.ipynb)

このノートブックは、教材のページのコードを順番に並べたものです（`tools/build_notebooks.py` で自動生成）。
上のセルから順に `Shift + Enter` で実行してください。

In [ ]:
# Colab のときだけ、教材リポジトリを取ってきて読み込めるようにする
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !git clone -q --depth 1 https://github.com/Asg1223/rcj-soccer-ml.git /content/rcj-soccer-ml
    sys.path.insert(0, "/content/rcj-soccer-ml")
print("Colab で実行中" if IN_COLAB else "ローカルで実行中")

## 3-1 ニューロンと層

説明は教材のページを見てください：https://asg1223.github.io/rcj-soccer-ml/part3-neural-net/neurons/

### 手を動かす 1：ReLU と 1 個のニューロン

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def relu(z):
    return np.maximum(0, z)

z = np.linspace(-3, 3, 100)
plt.plot(z, relu(z))
plt.xlabel("z")
plt.ylabel("ReLU(z)")
plt.show()

# 入力 3 個、ニューロン 1 個
x = np.array([0.8, 0.1, 0.0])       # センサ 3 個の値（0〜1）
w = np.array([1.0, -2.0, 0.5])      # 重み
b = -0.3                            # バイアス
print("ニューロンの出力:", relu(x @ w + b))

### 手を動かす 2：層を自分で書く

In [ ]:
def dense(x, W, b, activation=None):
    z = x @ W + b
    return relu(z) if activation == "relu" else z

rng = np.random.default_rng(0)
W1, b1 = rng.normal(0, 0.5, (12, 64)), np.zeros(64)    # 入力 12 → 64
W2, b2 = rng.normal(0, 0.2, (64, 64)), np.zeros(64)    # 64 → 64
W3, b3 = rng.normal(0, 0.2, (64, 4)), np.zeros(4)      # 64 → 出力 4

x = rng.random((5, 12))                 # センサの値 5 行分（でたらめ）
h1 = dense(x, W1, b1, "relu")
h2 = dense(h1, W2, b2, "relu")
y = dense(h2, W3, b3)
print(h1.shape, h2.shape, y.shape)      # (5, 64) (5, 64) (5, 4)
print("0 になったニューロンの割合（隠れ層1）:", (h1 == 0).mean().round(2))

### 手を動かす 3：重みの数を数える

In [ ]:
n_params = W1.size + b1.size + W2.size + b2.size + W3.size + b3.size
print("重みとバイアスの数:", n_params)
print("float（4 バイト）で保存すると:", n_params * 4 / 1024, "KB")

## 3-2 Keras で MLP：角度・距離・保持をまとめて推定する

説明は教材のページを見てください：https://asg1223.github.io/rcj-soccer-ml/part3-neural-net/keras-mlp/

### 手を動かす 1：データを用意する

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import keras
from rcjml import load_config
from rcjml.data import load_synthetic, split_by_session
from rcjml.nn import features, targets, decode, evaluate, keras_loss

cfg = load_config()
df = load_synthetic("ring12_large_normal")
train_all, test = split_by_session(df)            # テスト用：最後の 1/4 の取り直し

last = train_all["session"].max()
train = train_all[train_all["session"] < last]     # 学習用
val = train_all[train_all["session"] == last]      # 検証用：学習用の最後の取り直し

X_train, Y_train = features(train, cfg), targets(train)
X_val, Y_val = features(val, cfg), targets(val)
X_test = features(test, cfg)
print(X_train.shape, Y_train.shape)               # (4000, 12) (4000, 4)
print(Y_train[:3].round(2))                        # cos, sin, 距離, 保持

### 手を動かす 2：ネットを作る

In [ ]:
keras.utils.set_random_seed(0)      # 毎回同じ結果になるようにする

model = keras.Sequential([
    keras.Input(shape=(12,)),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dense(4),                    # 出力：活性化関数なし
])
model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss=keras_loss())
model.summary()

### 手を動かす 3：学習させる

In [ ]:
stop = keras.callbacks.EarlyStopping(patience=20, restore_best_weights=True)
history = model.fit(
    X_train, Y_train,
    validation_data=(X_val, Y_val),
    epochs=400, batch_size=64,
    callbacks=[stop], verbose=0,
)
print("学習したエポック数:", len(history.history["loss"]))

plt.plot(history.history["loss"], label="train")
plt.plot(history.history["val_loss"], label="validation")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.yscale("log")
plt.legend()
plt.show()

### 手を動かす 4：テスト用データで成績を見る

In [ ]:
P = model.predict(X_test, verbose=0)
result = evaluate(P, test)
for k, v in result.items():
    print(f"{k:18s} {v:.3f}")

### 手を動かす 5：距離の推定を見る

In [ ]:
d = decode(P)
plt.figure(figsize=(5, 5))
plt.scatter(test["distance_cm"], d["distance_cm"], s=4, alpha=0.4)
plt.plot([3, 180], [3, 180], color="tab:orange")
plt.xscale("log")
plt.yscale("log")
plt.xlabel("true distance [cm]")
plt.ylabel("predicted distance [cm]")
plt.show()

## 3-3 過学習とデータの量

説明は教材のページを見てください：https://asg1223.github.io/rcj-soccer-ml/part3-neural-net/overfitting/

### 手を動かす 1：わざと過学習させる

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import keras
from rcjml import load_config
from rcjml.data import load_synthetic, split_by_session
from rcjml.nn import features, targets, evaluate, keras_loss, build_ballnet

cfg = load_config()
df = load_synthetic("ring12_large_normal")
train_all, test = split_by_session(df)
val = train_all[train_all.session == 8]
one = train_all[train_all.session == 0]          # 500 行だけ

keras.utils.set_random_seed(0)
big = build_ballnet(hidden=(128, 128))
big.compile(optimizer=keras.optimizers.Adam(1e-3), loss=keras_loss())
h = big.fit(features(one, cfg), targets(one),
            validation_data=(features(val, cfg), targets(val)),
            epochs=500, batch_size=32, verbose=0)

plt.plot(h.history["loss"], label="train")
plt.plot(h.history["val_loss"], label="validation")
plt.yscale("log")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.legend()
plt.show()
best = int(np.argmin(h.history["val_loss"]))
print(f"学習用の損失（最後）: {h.history['loss'][-1]:.3f}")
print(f"検証用の損失（いちばん良いとき）: {min(h.history['val_loss']):.3f}（{best} エポック目）")
print(f"検証用の損失（最後）: {h.history['val_loss'][-1]:.3f}")

### 手を動かす 2：早期終了と正則化

In [ ]:
def fit_eval(train, val, test, hidden=(64, 64), epochs=400, l2=0.0, dropout=0.0,
             early_stop=True, batch_size=64, seed=0):
    keras.utils.set_random_seed(seed)
    model = build_ballnet(hidden=hidden, l2=l2, dropout=dropout)
    model.compile(optimizer=keras.optimizers.Adam(1e-3), loss=keras_loss())
    cb = [keras.callbacks.EarlyStopping(patience=20, restore_best_weights=True)] if early_stop else []
    h = model.fit(features(train, cfg), targets(train),
                  validation_data=(features(val, cfg), targets(val)),
                  epochs=epochs, batch_size=batch_size, callbacks=cb, verbose=0)
    r = evaluate(model.predict(features(test, cfg), verbose=0), test)
    r["epochs"] = len(h.history["loss"])
    return r

print("早期終了なし :", round(evaluate(big.predict(features(test, cfg), verbose=0), test)["angle_MAE"], 1))
r1 = fit_eval(one, val, test, hidden=(128, 128), epochs=500, batch_size=32)
r2 = fit_eval(one, val, test, hidden=(128, 128), epochs=500, batch_size=32, l2=1e-4, dropout=0.1)
print("早期終了あり :", round(r1["angle_MAE"], 1), f"（{r1['epochs']} エポックで停止）")
print("＋L2＋ドロップアウト:", round(r2["angle_MAE"], 1), f"（{r2['epochs']} エポックで停止）")

### 手を動かす 3：データの量を変える

In [ ]:
from rcjml.sim import make_dataset

rows = {}
for k in [1, 2, 4, 8]:
    sub = train_all[train_all.session < k]
    rows[f"{k} sessions ({len(sub)} rows)"] = fit_eval(sub, val, test)["angle_MAE"]

big_data = make_dataset(cfg, n=24000, difficulty="normal", sessions=48, seed=7)
sim_train = big_data[big_data.session < 44]
sim_val = big_data[(big_data.session >= 44) & (big_data.session < 46)]
rows[f"simulator ({len(sim_train)} rows)"] = fit_eval(sim_train, sim_val, test)["angle_MAE"]
pd.Series(rows).round(1)

## 3-4 分類：ボールを保持しているか

説明は教材のページを見てください：https://asg1223.github.io/rcj-soccer-ml/part3-neural-net/classification/

### 手を動かす 1：3-2 のモデルで確率を出す

In [ ]:
p_hold = decode(P)["p_hold"]
true_hold = test["has_ball"].values == 1
print("保持しているデータの割合:", true_hold.mean().round(3))

plt.hist(p_hold[~true_hold], bins=50, alpha=0.6, label="not holding")
plt.hist(p_hold[true_hold], bins=50, alpha=0.6, label="holding")
plt.yscale("log")
plt.xlabel("predicted probability of holding")
plt.legend()
plt.show()

### 手を動かす 2：混同行列を作る

In [ ]:
def confusion(true, pred):
    return pd.DataFrame(
        [[(true & pred).sum(), (true & ~pred).sum()],
         [(~true & pred).sum(), (~true & ~pred).sum()]],
        index=["true: holding", "true: not"], columns=["pred: holding", "pred: not"],
    )

pred = p_hold > 0.5
print(confusion(true_hold, pred))
print("見つけた割合:", (pred[true_hold]).mean().round(3))
print("空振りの割合:", (pred[~true_hold]).mean().round(4))

### 手を動かす 3：しきい値を動かす

In [ ]:
rows = []
for th in [0.1, 0.3, 0.5, 0.7, 0.9, 0.97]:
    pred = p_hold > th
    rows.append({"threshold": th,
                 "recall": pred[true_hold].mean(),
                 "false alarm": pred[~true_hold].mean()})
pd.DataFrame(rows).round(4)

## 3-5 データ拡張：センサリングを回して水増しする

説明は教材のページを見てください：https://asg1223.github.io/rcj-soccer-ml/part3-neural-net/augmentation/

### 手を動かす 1：回してみる

In [ ]:
import numpy as np
import pandas as pd
import keras
from rcjml import load_config
from rcjml.data import load_synthetic, split_by_session, sensor_columns
from rcjml.nn import features, targets, evaluate, keras_loss, build_ballnet, rotate_ring, augment_rotations

cfg = load_config()
df = load_synthetic("ring12_large_normal")
S = sensor_columns(df)

one = df[df.has_ball == 0].head(1)
print(one[S + ["angle_deg"]])
print(rotate_ring(one, 1)[S + ["angle_deg"]])       # 1 つずらすと、角度が +30°

### 手を動かす 2：学習用データが少ないときと多いときで比べる

In [ ]:
train_all, test = split_by_session(df)
val = train_all[train_all.session == 8]

def fit_eval(train, val, test, epochs=400, seed=0):
    keras.utils.set_random_seed(seed)
    model = build_ballnet()
    model.compile(optimizer=keras.optimizers.Adam(1e-3), loss=keras_loss())
    stop = keras.callbacks.EarlyStopping(patience=20, restore_best_weights=True)
    model.fit(features(train, cfg), targets(train),
              validation_data=(features(val, cfg), targets(val)),
              epochs=epochs, batch_size=64, callbacks=[stop], verbose=0)
    return evaluate(model.predict(features(test, cfg), verbose=0), test)

rows = {}
for name, k in [("3 sessions", 3), ("8 sessions", 8)]:
    sub = train_all[train_all.session < k]
    rows[(name, "original")] = fit_eval(sub, val, test)
    rows[(name, "rotated x12")] = fit_eval(augment_rotations(sub), val, test, epochs=200)
pd.DataFrame(rows).T[["angle_MAE", "dist_rel_median"]].round(3)